Module 3: Hệ Khuyến Nghị Cá Nhân Hóa
Phương pháp: ALS (Alternating Least Squares) - Implicit Collaborative Filtering  
Dữ liệu: Lịch sử nghe nhạc (user_id, track_name, artist_name, timestamp)  
Đặc điểm: Batch training từng file, lưu model checkpoint sau mỗi batch


1. Cài đặt thư viện

In [2]:
!pip install implicit scikit-learn scipy pandas numpy tqdm joblib -q
import warnings
warnings.filterwarnings('ignore')
print('Cài đặt hoàn tất')

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Cài đặt hoàn tất


2. Import & Cấu hình

In [6]:
import os, glob, gc, pickle, json, time
import numpy as np
import pandas as pd
import scipy.sparse as sp
from tqdm import tqdm
from collections import defaultdict
from implicit.als import AlternatingLeastSquares
from implicit.evaluation import train_test_split, mean_average_precision_at_k
import joblib

# ─── CẤU HÌNH ───────────────────────────────────────────────
CONFIG = {
    # Đường dẫn
    'data_dir'       : '/kaggle/input/datasets/js042710/second3t1k/CLEARDATA',   # ← thay đúng tên dataset Kaggle
    'output_dir'     : '/kaggle/working/model',
    'checkpoint_dir' : '/kaggle/working/checkpoints',

    # ALS Hyperparameters
    'factors'        : 128,      # số chiều latent
    'iterations'     : 20,       # số vòng lặp ALS
    'regularization' : 0.01,
    'alpha'          : 40,       # confidence weight cho implicit feedback
    'use_gpu'        : True,    # False nếu không GPU

    # Xử lý dữ liệu
    'min_interactions': 3,       # lọc user/item ít tương tác
    'item_key'        : 'track_artist',  # unique item = track+artist
    'chunk_size'      : 500_000, # đọc từng chunk trong file lớn
}

os.makedirs(CONFIG['output_dir'], exist_ok=True)
os.makedirs(CONFIG['checkpoint_dir'], exist_ok=True)
print('Thư mục đầu ra:', CONFIG['output_dir'])
print('Config:', json.dumps({k:v for k,v in CONFIG.items() if 'dir' not in k}, indent=2))

Thư mục đầu ra: /kaggle/working/model
Config: {
  "factors": 128,
  "iterations": 20,
  "regularization": 0.01,
  "alpha": 40,
  "use_gpu": true,
  "min_interactions": 3,
  "item_key": "track_artist",
  "chunk_size": 500000
}


3. Khám phá dữ liệu

In [7]:
# Lấy danh sách tất cả file CSV
all_files = sorted(glob.glob(os.path.join(CONFIG['data_dir'], '*.csv')))
# Thêm xlsx nếu có
all_files += sorted(glob.glob(os.path.join(CONFIG['data_dir'], '*.xlsx')))
print(f'Tổng số file: {len(all_files)}')
for f in all_files[:5]:
    print(' -', os.path.basename(f))

# Preview file đầu tiên
df_sample = pd.read_csv(all_files[0], nrows=5)
print('\nSchema:')
print(df_sample.dtypes)
print('\nSample:')
df_sample

Tổng số file: 30
 - clean_1.2.2026.listens.csv
 - clean_10.1.2026.listens.csv
 - clean_11.1.2026.listens.csv
 - clean_12.1.2026.listens.csv
 - clean_13.1.2026.listens.csv

Schema:
user_id            int64
timestamp         object
recording_msid    object
track_name        object
artist_name       object
release_name      object
dtype: object

Sample:


,user_id,timestamp,recording_msid,track_name,artist_name,release_name
0,1,2026-01-31 12:57:58,2d39940b-f575-42a7-b636-e3ab6880bbca,Auxin,Cell,Onwards System
1,1,2026-01-31 13:04:37,230a1f84-609d-43d1-a50e-bff4a34aff0c,Figment,Cell,Onwards System
2,1,2026-01-31 13:11:07,025df4fb-5bc4-4b34-814d-ec6aa6fe08aa,Geiger,Cell,Onwards System
3,5,2026-01-31 08:04:43,11fa2a40-66e4-40a0-bd4f-3a06c9848644,You Know How We Do (Camilo Dos Santos & Crafts...,Ice Cube,Chapter Eighteen Edits
4,5,2026-01-31 08:36:21,8d4f83c7-a6fc-497d-9be8-4ff457656c9d,House Down,Even Funkier,Chapter Eighteen Edits


4. Xây dựng Index (User & Item)

In [8]:
class InteractionAccumulator:
    """
    Tích lũy tương tác user-item từ nhiều file,
    xây dựng index và sparse matrix.
    """
    def __init__(self):
        self.user2idx  = {}   # user_id  → int index
        self.item2idx  = {}   # item_key → int index
        self.idx2user  = []
        self.idx2item  = []
        # {(user_idx, item_idx): count}
        self.interactions = defaultdict(int)
        self.n_rows_processed = 0

    def _get_or_create(self, mapping, reverse, key):
        if key not in mapping:
            mapping[key] = len(reverse)
            reverse.append(key)
        return mapping[key]

    def add_dataframe(self, df: pd.DataFrame):
        """Thêm một DataFrame vào accumulator."""
        # Tạo item_key = track_name + '|||' + artist_name
        df = df[['user_id', 'track_name', 'artist_name']].dropna()
        df['item_key'] = df['track_name'].str.strip() + '|||' + df['artist_name'].str.strip()

        for row in df.itertuples(index=False):
            uid = self._get_or_create(self.user2idx, self.idx2user, str(row.user_id))
            iid = self._get_or_create(self.item2idx, self.idx2item, row.item_key)
            self.interactions[(uid, iid)] += 1

        self.n_rows_processed += len(df)

    def build_matrix(self, min_interactions=3):
        """Tạo sparse user-item matrix (item×user cho ALS)."""
        print(f'Building matrix từ {len(self.interactions):,} cặp tương tác...')

        rows, cols, data = [], [], []
        for (uid, iid), cnt in self.interactions.items():
            rows.append(uid)
            cols.append(iid)
            data.append(cnt)

        # user × item
        n_users = len(self.idx2user)
        n_items = len(self.idx2item)
        user_item = sp.csr_matrix(
            (data, (rows, cols)),
            shape=(n_users, n_items), dtype=np.float32
        )

        # Lọc item và user ít tương tác
        if min_interactions > 1:
            item_counts = np.asarray(user_item.sum(axis=0)).flatten()
            user_counts = np.asarray(user_item.sum(axis=1)).flatten()
            valid_items = item_counts >= min_interactions
            valid_users = user_counts >= min_interactions
            user_item = user_item[valid_users][:, valid_items]
            self.idx2user = [u for u, v in zip(self.idx2user, valid_users) if v]
            self.idx2item = [i for i, v in zip(self.idx2item, valid_items) if v]
            self.user2idx = {u: i for i, u in enumerate(self.idx2user)}
            self.item2idx = {i: j for j, i in enumerate(self.idx2item)}
            print(f'   Sau lọc (min={min_interactions}): {user_item.shape[0]:,} users, {user_item.shape[1]:,} items')

        print(f'   Ma trận cuối: {user_item.shape[0]:,} × {user_item.shape[1]:,}')
        print(f'   Sparsity: {100*(1 - user_item.nnz / (user_item.shape[0]*user_item.shape[1])):.2f}%')
        return user_item

accumulator = InteractionAccumulator()
print('Khởi tạo AccumulatorAccumulator')

Khởi tạo AccumulatorAccumulator


5. Load dữ liệu theo Batch

In [9]:
CHECKPOINT_INDEX_PATH = os.path.join(CONFIG['checkpoint_dir'], 'accumulator_checkpoint.pkl')
PROGRESS_PATH = os.path.join(CONFIG['checkpoint_dir'], 'progress.json')

def save_accumulator_checkpoint(acc, processed_files):
    """Lưu checkpoint accumulator."""
    state = {
        'user2idx': acc.user2idx,
        'item2idx': acc.item2idx,
        'idx2user': acc.idx2user,
        'idx2item': acc.idx2item,
        'interactions': dict(acc.interactions),
        'n_rows_processed': acc.n_rows_processed,
    }
    joblib.dump(state, CHECKPOINT_INDEX_PATH, compress=3)
    with open(PROGRESS_PATH, 'w') as f:
        json.dump({'processed_files': processed_files}, f)
    print(f'   Checkpoint lưu: {len(processed_files)} files')

def load_accumulator_checkpoint():
    """Khôi phục từ checkpoint nếu tồn tại."""
    if not os.path.exists(CHECKPOINT_INDEX_PATH):
        return InteractionAccumulator(), []
    state = joblib.load(CHECKPOINT_INDEX_PATH)
    acc = InteractionAccumulator()
    acc.user2idx = state['user2idx']
    acc.item2idx = state['item2idx']
    acc.idx2user = state['idx2user']
    acc.idx2item = state['idx2item']
    acc.interactions = defaultdict(int, state['interactions'])
    acc.n_rows_processed = state['n_rows_processed']
    with open(PROGRESS_PATH) as f:
        progress = json.load(f)
    print(f'Khôi phục từ checkpoint: {len(progress["processed_files"])} files đã xử lý')
    return acc, progress['processed_files']

# ─── LOAD DỮ LIỆU ──────────────────────────────────────────
RESUME_FROM_CHECKPOINT = True  # ← False để train lại từ đầu

if RESUME_FROM_CHECKPOINT:
    accumulator, processed_files = load_accumulator_checkpoint()
else:
    accumulator, processed_files = InteractionAccumulator(), []

remaining_files = [f for f in all_files if f not in processed_files]
print(f'Còn {len(remaining_files)} files cần xử lý')

Còn 30 files cần xử lý


In [10]:
SAVE_EVERY_N_FILES = 5   # Lưu checkpoint sau mỗi N file

t0 = time.time()
for i, fpath in enumerate(tqdm(remaining_files, desc='Loading files')):
    fname = os.path.basename(fpath)
    try:
        if fpath.endswith('.xlsx'):
            df = pd.read_excel(fpath, usecols=['user_id','track_name','artist_name'])
            accumulator.add_dataframe(df)
        else:
            # Đọc theo chunk để tiết kiệm RAM
            for chunk in pd.read_csv(
                fpath,
                usecols=['user_id','track_name','artist_name'],
                chunksize=CONFIG['chunk_size'],
                low_memory=False
            ):
                accumulator.add_dataframe(chunk)
                del chunk
                gc.collect()

        processed_files.append(fpath)

        # Lưu checkpoint định kỳ
        if (i + 1) % SAVE_EVERY_N_FILES == 0:
            save_accumulator_checkpoint(accumulator, processed_files)

    except Exception as e:
        print(f'Lỗi file {fname}: {e}')
        continue

# Lưu lần cuối
save_accumulator_checkpoint(accumulator, processed_files)

elapsed = time.time() - t0
print(f'\nXử lý xong {len(processed_files)} files trong {elapsed/60:.1f} phút')
print(f'   Tổng số dòng: {accumulator.n_rows_processed:,}')
print(f'   Users: {len(accumulator.idx2user):,}')
print(f'   Items: {len(accumulator.idx2item):,}')

Loading files:  17%|█▋        | 5/30 [01:37<10:40, 25.62s/it]

   Checkpoint lưu: 5 files


Loading files:  33%|███▎      | 10/30 [04:04<13:30, 40.53s/it]

   Checkpoint lưu: 10 files


Loading files:  50%|█████     | 15/30 [07:30<14:48, 59.20s/it]

...

Loading files:  83%|████████▎ | 25/30 [15:34<06:39, 79.85s/it]

   Checkpoint lưu: 25 files


Loading files: 100%|██████████| 30/30 [21:19<00:00, 42.63s/it] 

   Checkpoint lưu: 30 files


   Checkpoint lưu: 30 files

Xử lý xong 30 files trong 24.7 phút
   Tổng số dòng: 81,267,084
   Users: 23,391
   Items: 8,799,921


6. Xây dựng Sparse Matrix & Train ALS

In [11]:
# Xây dựng user-item matrix
user_item_matrix = accumulator.build_matrix(min_interactions=CONFIG['min_interactions'])

# ALS yêu cầu item × user cho training
item_user_matrix = user_item_matrix.T.tocsr()

print(f'\nSẵn sàng train ALS')
print(f'   item×user matrix: {item_user_matrix.shape}')

Building matrix từ 23,660,619 cặp tương tác...
   Sau lọc (min=3): 22,087 users, 3,295,075 items
   Ma trận cuối: 22,087 × 3,295,075
   Sparsity: 99.98%

Sẵn sàng train ALS
   item×user matrix: (3295075, 22087)


In [12]:
# ─── TRAIN/TEST SPLIT ──────────────────────────────────────
train_matrix, test_matrix = train_test_split(
    user_item_matrix, train_percentage=0.8
)
train_item_user = train_matrix.T.tocsr()
print(f'Train: {train_matrix.nnz:,} interactions')
print(f'Test:  {test_matrix.nnz:,} interactions')

Train: 14,169,129 interactions
Test:  3,538,749 interactions


In [13]:
# ─── TRAIN ALS MODEL ───────────────────────────────────────
MODEL_CHECKPOINT = os.path.join(CONFIG['checkpoint_dir'], 'als_model_latest.pkl')

RESUME_MODEL = True  # ← False để train lại model từ đầu

if RESUME_MODEL and os.path.exists(MODEL_CHECKPOINT):
    print('Load model từ checkpoint...')
    model = joblib.load(MODEL_CHECKPOINT)
    print('Model loaded')
else:
    print('Bắt đầu train ALS...')
    model = AlternatingLeastSquares(
        factors        = CONFIG['factors'],
        iterations     = CONFIG['iterations'],
        regularization = CONFIG['regularization'],
        alpha          = CONFIG['alpha'],
        use_gpu        = CONFIG['use_gpu'],
        calculate_training_loss=True,
        random_state   = 42,
    )

    t0 = time.time()
    model.fit(train_item_user)
    elapsed = time.time() - t0

    print(f'Train xong trong {elapsed/60:.1f} phút')

    # Lưu model checkpoint
    joblib.dump(model, MODEL_CHECKPOINT, compress=3)
    print(f'Đã lưu model: {MODEL_CHECKPOINT}')

Bắt đầu train ALS...


  0%|          | 0/20 [00:00<?, ?it/s]

Train xong trong 0.5 phút
Đã lưu model: /kaggle/working/checkpoints/als_model_latest.pkl


7. Đánh giá Model

In [15]:
# ─── ĐÁNH GIÁ MODEL
import numpy as np

def evaluate_model(model, train_mat, test_mat, K=10, n_users=2000):
    """
    Tính MAP@K, Precision@K, Recall@K thủ công.
    Tương thích với mọi phiên bản scipy/implicit.
    """
    rng = np.random.default_rng(42)
    n_eval = min(n_users, test_mat.shape[0])
    test_users = rng.choice(test_mat.shape[0], n_eval, replace=False)

    ap_list, prec_list, rec_list = [], [], []

    for u in test_users:
        actual = set(test_mat[u].indices)
        if not actual:
            continue

        rec_ids, _ = model.recommend(
            u, train_mat[u], N=K,
            filter_already_liked_items=True
        )
        rec_ids = list(rec_ids)

        # Precision@K
        hits = len(set(rec_ids) & actual)
        prec_list.append(hits / K)

        # Recall@K
        rec_list.append(hits / len(actual))

        # AP@K
        ap, n_hits = 0.0, 0
        for rank, item in enumerate(rec_ids, 1):
            if item in actual:
                n_hits += 1
                ap += n_hits / rank
        ap_list.append(ap / min(len(actual), K))

    print(f'Kết quả đánh giá (n_users={n_eval}, K={K}):')
    print(f'   MAP@{K}       : {np.mean(ap_list):.4f}')
    print(f'   Precision@{K} : {np.mean(prec_list):.4f}')
    print(f'   Recall@{K}    : {np.mean(rec_list):.4f}')
    return np.mean(ap_list), np.mean(prec_list), np.mean(rec_list)

map_at_10, p10, r10 = evaluate_model(model, train_matrix, test_matrix, K=10)
_, p20, r20          = evaluate_model(model, train_matrix, test_matrix, K=20)

Kết quả đánh giá (n_users=2000, K=10):
   MAP@10       : 0.0001
   Precision@10 : 0.0003
   Recall@10    : 0.0000
Kết quả đánh giá (n_users=2000, K=20):
   MAP@20       : 0.0001
   Precision@20 : 0.0003
   Recall@20    : 0.0001


8. Lưu Model Hoàn Chỉnh

In [16]:
FINAL_MODEL_DIR = CONFIG['output_dir']

# 1. Lưu ALS model
joblib.dump(model, os.path.join(FINAL_MODEL_DIR, 'als_model.pkl'), compress=3)

# 2. Lưu index mappings
joblib.dump({
    'user2idx': accumulator.user2idx,
    'item2idx': accumulator.item2idx,
    'idx2user': accumulator.idx2user,
    'idx2item': accumulator.idx2item,
}, os.path.join(FINAL_MODEL_DIR, 'index_mappings.pkl'), compress=3)

# 3. Lưu user-item sparse matrix (để reuse)
sp.save_npz(os.path.join(FINAL_MODEL_DIR, 'user_item_matrix.npz'), user_item_matrix)

# 4. Lưu item metadata (track_name + artist_name)
item_meta = pd.DataFrame([
    {'item_idx': i, 'track_name': k.split('|||')[0], 'artist_name': k.split('|||')[1]}
    for i, k in enumerate(accumulator.idx2item)
])
item_meta.to_parquet(os.path.join(FINAL_MODEL_DIR, 'item_metadata.parquet'), index=False)

# 5. Lưu config
with open(os.path.join(FINAL_MODEL_DIR, 'config.json'), 'w') as f:
    json.dump({
        **CONFIG,
        'n_users': len(accumulator.idx2user),
        'n_items': len(accumulator.idx2item),
        'map_at_10': float(map_at_10),
        'precision_at_10': float(p10),
    }, f, indent=2)

print('Đã lưu đầy đủ:')
for f in os.listdir(FINAL_MODEL_DIR):
    fpath = os.path.join(FINAL_MODEL_DIR, f)
    size_mb = os.path.getsize(fpath) / 1e6
    print(f'   {f:40s} {size_mb:8.2f} MB')

Đã lưu đầy đủ:
   als_model.pkl                             1332.52 MB
   item_metadata.parquet                       96.16 MB
   index_mappings.pkl                          72.52 MB
   config.json                                  0.00 MB
   user_item_matrix.npz                        46.47 MB


9. Hàm Khuyến Nghị

In [30]:
class MusicRecommender:
    """
    Hệ khuyến nghị nhạc cá nhân hóa dựa trên ALS.
    Hỗ trợ: recommend by user_id, similar tracks, similar users.
    """

    def __init__(self, model_dir: str):
        self.model     = joblib.load(os.path.join(model_dir, 'als_model.pkl'))
        mappings       = joblib.load(os.path.join(model_dir, 'index_mappings.pkl'))
        self.user2idx  = mappings['user2idx']
        self.item2idx  = mappings['item2idx']
        self.idx2user  = mappings['idx2user']
        self.idx2item  = mappings['idx2item']
        self.user_item = sp.load_npz(os.path.join(model_dir, 'user_item_matrix.npz'))
        self.item_meta = pd.read_parquet(os.path.join(model_dir, 'item_metadata.parquet'))
        print(f'✅ Model loaded | {len(self.idx2user):,} users | {len(self.idx2item):,} items')

    def _item_idx_to_df(self, item_ids, scores):
        """Chuyển item index sang DataFrame đẹp."""
        rows = []
        for idx, score in zip(item_ids, scores):
            key = self.idx2item[idx]
            track, artist = key.split('|||')
            rows.append({'track_name': track, 'artist_name': artist, 'score': round(float(score), 4)})
        return pd.DataFrame(rows)

    def recommend_for_user(self, user_id, n=10, filter_listened=True):
        """
        Khuyến nghị bài hát cho user dựa trên lịch sử nghe.
        user_id: int hoặc str theo dữ liệu gốc.
        """
        uid_str = str(user_id)
        if uid_str not in self.user2idx:
            print(f'⚠️ User {user_id} không có trong tập train (cold-start).')
            return self.popular_items(n)

        uid = self.user2idx[uid_str]
        rec_ids, scores = self.model.recommend(
            uid,
            self.user_item[uid],
            N=n,
            filter_already_liked_items=filter_listened
        )
        df = self._item_idx_to_df(rec_ids, scores)
        df.index = range(1, len(df)+1)
        return df

    def similar_tracks(self, track_name, artist_name, n=10):
        """
        Tìm bài hát tương tự dựa trên latent factor.
        """
        key = f'{track_name.strip()}|||{artist_name.strip()}'
        if key not in self.item2idx:
            print(f'⚠️ Không tìm thấy: "{track_name}" - {artist_name}')
            return pd.DataFrame()

        item_idx = self.item2idx[key]
        similar_ids, scores = self.model.similar_items(item_idx, N=n+1)
        # Bỏ chính bài đó
        similar_ids = [i for i in similar_ids if i != item_idx][:n]
        scores = scores[1:n+1]
        df = self._item_idx_to_df(similar_ids, scores)
        df.index = range(1, len(df)+1)
        return df

    def similar_users(self, user_id, n=10):
        """
        Tìm users có gu nhạc tương tự.
        """
        uid_str = str(user_id)
        if uid_str not in self.user2idx:
            return pd.DataFrame()
        uid = self.user2idx[uid_str]
        sim_ids, scores = self.model.similar_users(uid, N=n+1)
        sim_ids = [i for i in sim_ids if i != uid][:n]
        df = pd.DataFrame({
            'user_id': [self.idx2user[i] for i in sim_ids],
            'similarity': [round(float(s),4) for s in scores[1:n+1]]
        })
        df.index = range(1, len(df)+1)
        return df

    def popular_items(self, n=10):
        """
        Fallback cold-start: trả về bài hát phổ biến nhất.
        """
        item_counts = np.asarray(self.user_item.sum(axis=0)).flatten()
        top_ids = np.argsort(item_counts)[::-1][:n]
        df = self._item_idx_to_df(top_ids, item_counts[top_ids])
        df.rename(columns={'score': 'play_count'}, inplace=True)
        df.index = range(1, len(df)+1)
        return df

    def user_history(self, user_id, n=20):
        """Xem lịch sử nghe của user."""
        uid_str = str(user_id)
        if uid_str not in self.user2idx:
            return pd.DataFrame()
        uid = self.user2idx[uid_str]
        row = self.user_item[uid]
        item_ids = row.indices
        counts = row.data
        top_n = np.argsort(counts)[::-1][:n]
        df = self._item_idx_to_df(item_ids[top_n], counts[top_n])
        df.rename(columns={'score': 'play_count'}, inplace=True)
        df.index = range(1, len(df)+1)
        return df

print('✅ Class MusicRecommender định nghĩa xong')

✅ Class MusicRecommender định nghĩa xong


10. Demo & Test

In [31]:
# Load recommender
rec = MusicRecommender(CONFIG['output_dir'])

# ─── Test với user thực tế ─────────────────────────────────
TEST_USER_ID = 1   # ← thay bằng user_id bất kỳ

print(f'\nLịch sử nghe của User {TEST_USER_ID}:')
display(rec.user_history(TEST_USER_ID, n=10))

print(f'\nTop 10 khuyến nghị cho User {TEST_USER_ID}:')
display(rec.recommend_for_user(TEST_USER_ID, n=10))

MemoryError: std::bad_alloc: out_of_memory: CUDA error at: /tmp/pip-install-lg64uxd4/implicit_fc6fbabf05834cf7a3fde5c2e422957b/_skbuild/linux-x86_64-3.12/cmake-build/_deps/rmm-src/include/rmm/mr/device/cuda_memory_resource.hpp:70: cudaErrorMemoryAllocation out of memory

In [23]:
# ─── Bài hát tương tự ─────────────────────────────────────
print('Bài hát tương tự "Auxin" - Cell:')
display(rec.similar_tracks('Auxin', 'Cell', n=10))

# ─── User có gu tương tự ──────────────────────────────────
print(f'\nUsers có gu nhạc giống User {TEST_USER_ID}:')
display(rec.similar_users(TEST_USER_ID, n=5))

# ─── Cold-start: bài phổ biến nhất ────────────────────────
print('\nTop bài hát phổ biến nhất (Cold-start fallback):')
display(rec.popular_items(n=10))

Bài hát tương tự "Auxin" - Cell:


,track_name,artist_name,score
1,Swift Glide,Optical,0.7103
2,Glitter,Say Lou Lou,0.6775
3,Ohrwurm,Cephalic Carnage,0.6765
4,Love Me Do - Remastered 2009,The Beatles,0.6687
5,Titanic,New End Original,0.6545
6,Sell Out,Reel Big Fish,0.6535
7,Azukiiro No Kaori,Susumu Yokota,0.6494
8,Chains,Tina Arena,0.6458
9,Downtown,Petula Clark,0.6405
10,FI3AC2512402,Aleksi Perälä,0.6296



Users có gu nhạc giống User 1:


IndexError: list index out of range

11. Fine-tune / Update Model với dữ liệu mới:
Khi có thêm file dữ liệu mới, chạy lại cell dưới để update model mà không cần train từ đầu.

In [24]:
def update_model_with_new_data(new_file_paths: list, model_dir: str, n_iter_update=10):
    """
    Cập nhật ALS model với dữ liệu mới.
    Tích lũy thêm vào accumulator hiện có, re-fit model.
    """
    # Load accumulator cũ
    old_acc, old_processed = load_accumulator_checkpoint()

    # Thêm dữ liệu mới
    for fpath in tqdm(new_file_paths, desc='Loading new data'):
        for chunk in pd.read_csv(
            fpath,
            usecols=['user_id','track_name','artist_name'],
            chunksize=500_000
        ):
            old_acc.add_dataframe(chunk)

    # Rebuild matrix
    new_matrix = old_acc.build_matrix(min_interactions=CONFIG['min_interactions'])

    # Re-fit model (warm start bằng cách dùng ít iterations hơn)
    new_model = AlternatingLeastSquares(
        factors=CONFIG['factors'],
        iterations=n_iter_update,
        regularization=CONFIG['regularization'],
        alpha=CONFIG['alpha'],
        use_gpu=CONFIG['use_gpu'],
        random_state=42,
    )
    new_model.fit(new_matrix.T.tocsr())

    # Lưu
    joblib.dump(new_model, os.path.join(model_dir, 'als_model.pkl'), compress=3)
    joblib.dump({
        'user2idx': old_acc.user2idx, 'item2idx': old_acc.item2idx,
        'idx2user': old_acc.idx2user, 'idx2item': old_acc.idx2item,
    }, os.path.join(model_dir, 'index_mappings.pkl'), compress=3)
    sp.save_npz(os.path.join(model_dir, 'user_item_matrix.npz'), new_matrix)
    save_accumulator_checkpoint(old_acc, old_processed + new_file_paths)

    print(f'Model đã được cập nhật với {len(new_file_paths)} file mới')
    return new_model, old_acc

# Ví dụ:
# update_model_with_new_data(['/kaggle/input/newdata/clean_1.3.2026.csv'], CONFIG['output_dir'])
print('Hàm update_model_with_new_data đã sẵn sàng')

Hàm update_model_with_new_data đã sẵn sàng


Tóm tắt File Output

| File | Mô tả |
|------|-------|
| `als_model.pkl` | ALS model chính |
| `index_mappings.pkl` | user2idx, item2idx, idx2user, idx2item |
| `user_item_matrix.npz` | Sparse matrix (để recommend) |
| `item_metadata.parquet` | Metadata bài hát |
| `config.json` | Config + metrics |
| `checkpoints/` | Checkpoint trung gian |
